## Практическая работа 3 — Векторные представления (C#)

- Единое предобработанное ядро текстов (нормализация, токенизация, стоп-слова)
- Три способа представления документов: LSA, Word2Vec, Doc2Vec
- Логистическая регрессия и метрики качества
- Сравнение, поиск похожих документов, визуализации

Работа продолжает ПР2: на той же коллекции отзывов сравниваем классические распределённые представления без использования нейросетей семейства RNN/CNN.


In [13]:
#r "nuget: Microsoft.Data.Analysis, 0.22.2"
#r "nuget: Microsoft.ML, 3.0.1"
#r "nuget: Microsoft.ML.Mkl.Components, 3.0.1"
#r "nuget: ScottPlot, 5.0.56"
#r "nuget: System.Text.Encoding.CodePages, 8.0.0"

Installed Packages Microsoft.Data.Analysis, 0.22.2 Microsoft.ML, 3.0.1 Microsoft.ML.Mkl.Components, 3.0.1 ScottPlot, 5.0.56 System.Text.Encoding.CodePages, 8.0.0

In [14]:
using System;
using System.IO;
using System.Text;
using System.Text.RegularExpressions;
using System.Collections.Generic;
using System.Linq;
using System.Diagnostics;

using Microsoft.Data.Analysis;
using Microsoft.ML;
using Microsoft.ML.Data;
using Microsoft.ML.Transforms.Text;

using ScottPlot;
using Microsoft.DotNet.Interactive.Formatting;

Encoding.RegisterProvider(CodePagesEncodingProvider.Instance);

Formatter.Register<DataFrame>((df, writer) => writer.Write(df.ToString()), "text/plain");
Formatter.Register(typeof(ScottPlot.Plot),
    (obj, writer) => writer.Write(((ScottPlot.Plot)obj).GetPngHtml(900, 520)),
    HtmlFormatter.MimeType);

var mlContext = new MLContext(seed: 42);

double Sigmoid(double x) => 1.0 / (1.0 + Math.Exp(-x));

double Cosine(float[] a, float[] b)
{
    double dot = 0, na = 0, nb = 0;
    int len = Math.Min(a.Length, b.Length);
    for (int i = 0; i < len; i++)
    {
        double aa = a[i];
        double bb = b[i];
        dot += aa * bb;
        na += aa * aa;
        nb += bb * bb;
    }
    return na > 0 && nb > 0 ? dot / Math.Sqrt(na * nb) : 0.0;
}

void NormalizeInPlace(float[] vector)
{
    double sumSq = 0;
    for (int i = 0; i < vector.Length; i++)
        sumSq += vector[i] * vector[i];
    if (sumSq <= 0) return;
    float scale = (float)(1.0 / Math.Sqrt(sumSq));
    for (int i = 0; i < vector.Length; i++)
        vector[i] *= scale;
}

float[] Clone(float[] source)
{
    var copy = new float[source.Length];
    Array.Copy(source, copy, source.Length);
    return copy;
}

float[] ToDense(VBuffer<float> buffer)
{
    var dense = new float[buffer.Length];
    buffer.CopyTo(dense);
    return dense;
}

int SampleNegative(Random random, double[] cumulative)
{
    var value = random.NextDouble();
    int idx = Array.BinarySearch(cumulative, value);
    if (idx < 0) idx = ~idx;
    if (idx >= cumulative.Length) idx = cumulative.Length - 1;
    return idx;
}

void Shuffle<T>(IList<T> list, Random random)
{
    for (int i = list.Count - 1; i > 0; i--)
    {
        int j = random.Next(i + 1);
        (list[i], list[j]) = (list[j], list[i]);
    }
}

string Truncate(string text, int maxLength = 160)
{
    if (string.IsNullOrWhiteSpace(text)) return string.Empty;
    return text.Length <= maxLength ? text : text.Substring(0, maxLength) + "…";
}

List<(int index, double score)> GetTopSimilar(float[] query, IReadOnlyList<float[]> candidates, int top = 5)
{
    var list = new List<(int index, double score)>(candidates.Count);
    for (int i = 0; i < candidates.Count; i++)
    {
        var score = Cosine(query, candidates[i]);
        if (double.IsNaN(score))
            continue;
        list.Add((index: i, score: score));
    }
    return list
        .OrderByDescending(item => item.score)
        .Take(top)
        .ToList();
}

int[] SampleIndices(int length, int maxSample, Random random)
{
    if (length <= maxSample)
        return Enumerable.Range(0, length).ToArray();

    var reservoir = new int[maxSample];
    for (int i = 0; i < maxSample; i++)
        reservoir[i] = i;

    for (int i = maxSample; i < length; i++)
    {
        int j = random.Next(i + 1);
        if (j < maxSample)
            reservoir[j] = i;
    }

    Array.Sort(reservoir);
    return reservoir;
}


### 1. Загрузка и первичная очистка

Загружаем csv из ПР2, убираем строки с пропусками и нейтральной тональностью.
Сохраняем форму датафреймов, чтобы контролировать, что в дальнейшие шаги попадает сопоставимый корпус.


In [15]:
DataFrame LoadDf(string path, string encoding = "windows-1251")
{
    using var fs = File.OpenRead(path);
    return DataFrame.LoadCsv(fs, separator: ',', header: true, guessRows: 50_000, addIndexColumn: false, encoding: Encoding.GetEncoding(encoding));
}

var dfTrainRaw = LoadDf("../data/train.csv");
var dfTestRaw  = LoadDf("../data/test.csv");

DataFrame Clean(DataFrame df)
{
    var sentimentCol = df.Columns["sentiment"] as StringDataFrameColumn;
    var notNeutral = sentimentCol.ElementwiseNotEquals("neutral");
    return df.Filter(notNeutral);
}

var dfTrain = Clean(dfTrainRaw);
var dfTest  = Clean(dfTestRaw);

Console.WriteLine($"Train raw: {dfTrainRaw.Rows.Count}, after filter: {dfTrain.Rows.Count}");
Console.WriteLine($"Test  raw: {dfTestRaw.Rows.Count}, after filter: {dfTest.Rows.Count}");

dfTrain.Head(3)


Train raw: 27481, after filter: 16363
Test  raw: 4815, after filter: 3385


index,textID,text,selected_text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (KmІ),Density (P/KmІ)
0,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative,noon,21-30,Albania,2877797,27400,105
1,088c60f138,my boss is bullying me...,bullying me,negative,night,31-45,Algeria,43851044,2381740,18
2,9642c003ef,what interview! leave me alone,leave me alone,negative,morning,46-60,Andorra,77265,470,164


### 2. Нормализация, токенизация и корпус

Единый текстовый конвейер на ML.NET приводит символы к нижнему регистру, удаляет шум и выделяет токены.
Список `processedRecords` становится общим входом для всех последующих векторных представлений, поэтому сравнение делаем ровно на одном и том же наборе документов.


In [16]:
public class SentimentRecord
{
    public int DocId { get; set; }
    public string Text { get; set; }
    public string Sentiment { get; set; }
    public bool Label { get; set; }
    public string Split { get; set; }
}

public class TokenizedRecord : SentimentRecord
{
    public string NormalizedText { get; set; }
    public ReadOnlyMemory<char>[] TokensRaw { get; set; }
    public ReadOnlyMemory<char>[] Tokens { get; set; }
}

public class ProcessedRecord
{
    public int GlobalId { get; set; }
    public bool IsTrain { get; set; }
    public int SplitIndex { get; set; }
    public string Text { get; set; }
    public string CleanText { get; set; }
    public string Sentiment { get; set; }
    public bool Label { get; set; }
    public string[] Tokens { get; set; }
}

var allRecords = new List<SentimentRecord>();
int docCounter = 0;

void AddRecords(DataFrame df, string split)
{
    var textCol = df.Columns["text"];
    var sentimentCol = df.Columns["sentiment"];
    for (int i = 0; i < df.Rows.Count; i++)
    {
        var text = textCol[i]?.ToString();
        var sentiment = sentimentCol[i]?.ToString();
        if (string.IsNullOrWhiteSpace(text) || string.IsNullOrWhiteSpace(sentiment))
            continue;
        allRecords.Add(new SentimentRecord
        {
            DocId = docCounter++,
            Text = text,
            Sentiment = sentiment,
            Label = sentiment.Equals("positive", StringComparison.OrdinalIgnoreCase),
            Split = split
        });
    }
}

AddRecords(dfTrain, "train");
AddRecords(dfTest,  "test");

Console.WriteLine($"Records after cleanup: {allRecords.Count} (train: {allRecords.Count(r => r.Split == "train")}, test: {allRecords.Count(r => r.Split == "test")})");

var allDataView = mlContext.Data.LoadFromEnumerable(allRecords);
var textPipeline = mlContext.Transforms.Text.NormalizeText(
        outputColumnName: "NormalizedText",
        inputColumnName: nameof(SentimentRecord.Text),
        caseMode: TextNormalizingEstimator.CaseMode.Lower,
        keepDiacritics: false,
        keepNumbers: false,
        keepPunctuations: false)
    .Append(mlContext.Transforms.Text.TokenizeIntoWords(
        outputColumnName: "TokensRaw",
        inputColumnName: "NormalizedText"))
    .Append(mlContext.Transforms.Text.RemoveDefaultStopWords(
        outputColumnName: "Tokens",
        inputColumnName: "TokensRaw",
        language: StopWordsRemovingEstimator.Language.English));

var textTransformer = textPipeline.Fit(allDataView);
var tokenizedView = textTransformer.Transform(allDataView);
var tokenized = mlContext.Data.CreateEnumerable<TokenizedRecord>(tokenizedView, reuseRowObject: false).ToList();

List<ProcessedRecord> processedRecords = new();
List<ProcessedRecord> trainRecords = new();
List<ProcessedRecord> testRecords = new();

foreach (var rec in tokenized)
{
    var tokens = (rec.Tokens ?? Array.Empty<ReadOnlyMemory<char>>())
        .Select(t => t.ToString())
        .Where(t => !string.IsNullOrWhiteSpace(t) && t.Length > 1)
        .ToArray();

    if (tokens.Length < 3)
        continue;

    bool isTrain = rec.Split == "train";
    var processed = new ProcessedRecord
    {
        IsTrain = isTrain,
        Text = rec.Text,
        CleanText = string.Join(" ", tokens),
        Sentiment = rec.Sentiment,
        Label = rec.Label,
        Tokens = tokens
    };
    processedRecords.Add(processed);
    if (isTrain)
        trainRecords.Add(processed);
    else
        testRecords.Add(processed);
}

int maxTrainDocs = 4000;
if (trainRecords.Count > maxTrainDocs)
{
    var rng = new Random(42);
    Shuffle(trainRecords, rng);
    trainRecords = trainRecords.Take(maxTrainDocs).ToList();
}

var reordered = new List<ProcessedRecord>();
int globalIdx = 0;
for (int i = 0; i < trainRecords.Count; i++)
{
    var rec = trainRecords[i];
    rec.GlobalId = globalIdx++;
    rec.SplitIndex = i;
    rec.IsTrain = true;
    reordered.Add(rec);
}
for (int i = 0; i < testRecords.Count; i++)
{
    var rec = testRecords[i];
    rec.GlobalId = globalIdx++;
    rec.SplitIndex = i;
    rec.IsTrain = false;
    reordered.Add(rec);
}
processedRecords = reordered;

Console.WriteLine($"Train docs after preprocessing: {trainRecords.Count}");
Console.WriteLine($"Test  docs after preprocessing: {testRecords.Count}");

foreach (var example in trainRecords.Take(3))
{
    Console.WriteLine("---");
    Console.WriteLine(example.Text);
    Console.WriteLine($"Clean → {example.CleanText}");
}

public class ModelInput
{
    public string Text { get; set; }
    public bool Label { get; set; }
}

public class VectorExample
{
    public bool Label { get; set; }
    [VectorType(128)]
    public float[] Features { get; set; }
}


public class LsaVectorExample
{
    public bool Label { get; set; }
    [VectorType(300)]
    public float[] Features { get; set; } = Array.Empty<float>();
}

var trainInputs = trainRecords.Select(r => new ModelInput { Text = r.CleanText, Label = r.Label }).ToList();
var testInputs  = testRecords.Select(r => new ModelInput { Text = r.CleanText, Label = r.Label }).ToList();

var embeddingTrainer = mlContext.BinaryClassification.Trainers.LbfgsLogisticRegression(
    labelColumnName: nameof(VectorExample.Label),
    featureColumnName: nameof(VectorExample.Features));

BinaryClassificationMetrics? metricsTrainWord2Vec = null;
BinaryClassificationMetrics? metricsTestWord2Vec = null;
BinaryClassificationMetrics? metricsTrainDoc2Vec = null;
BinaryClassificationMetrics? metricsTestDoc2Vec = null;
ITransformer? word2VecModel = null;
ITransformer? doc2VecModel = null;
float[][] word2VecInputEmbeddings = Array.Empty<float[]>();
float[][] word2VecOutputEmbeddings = Array.Empty<float[]>();
float[][] doc2VecTrainedEmbeddings = Array.Empty<float[]>();
float[][] doc2VecWordOutput = Array.Empty<float[]>();
int word2VecDimension = 128;
int doc2VecDimension = 128;
int doc2VecNegativeSamples = 4;
double doc2VecInitialLearningRate = 0.05;
IDataView? predTrainWord2VecView = null;
IDataView? predTestWord2VecView = null;
IDataView? predTrainDoc2VecView = null;
IDataView? predTestDoc2VecView = null;
float[][] word2VecTrainMatrix = Array.Empty<float[]>();
float[][] word2VecTestMatrix = Array.Empty<float[]>();
float[][] doc2VecTrainMatrix = Array.Empty<float[]>();
float[][] doc2VecTestMatrix = Array.Empty<float[]>();



Records after cleanup: 18467 (train: 16363, test: 2104)
Train docs after preprocessing: 4000
Test  docs after preprocessing: 1930
---
Happy May Bank Holiday British Peeps
Clean → happy bank holiday british peeps
---
Awww  You`ll Be Fine...
Clean → awww you`ll fine
---
I believe a man died in a car wreck today just right down the road.  It happened at 12 and at 2 he was still in the car.
Clean → believe man died car wreck today just right road happened car


### 3. LSA: TF-IDF + PCA + нормализация

Строим TF-IDF по объединённому корпусу (train+test), затем через PCA снижаем размерность до 300 компонент и нормализуем векторы.
После трансформации обучаем логистическую регрессию и сохраняем dense-представления для дальнейших визуализаций и поиска ближайших.


In [17]:
int lsaRank = 300;

var allInputs = trainInputs.Concat(testInputs).ToList();
var allInputsView = mlContext.Data.LoadFromEnumerable(allInputs);

var lsaPipeline = mlContext.Transforms.Text.FeaturizeText(
        outputColumnName: "FeaturesRaw",
        inputColumnName: nameof(ModelInput.Text))
    .Append(mlContext.Transforms.ProjectToPrincipalComponents(
        outputColumnName: "FeaturesPca",
        inputColumnName: "FeaturesRaw",
        rank: lsaRank,
        ensureZeroMean: false))
    .Append(mlContext.Transforms.NormalizeLpNorm(
        outputColumnName: "Features",
        inputColumnName: "FeaturesPca"));

var lsaTransformer = lsaPipeline.Fit(allInputsView);

var trainLsaView = lsaTransformer.Transform(mlContext.Data.LoadFromEnumerable(trainInputs));
var testLsaView  = lsaTransformer.Transform(mlContext.Data.LoadFromEnumerable(testInputs));

var lsaTrainer = mlContext.BinaryClassification.Trainers.LbfgsLogisticRegression(
    labelColumnName: nameof(ModelInput.Label),
    featureColumnName: "Features");

var lsaModel = lsaTrainer.Fit(trainLsaView);
var predTrainLsa = lsaModel.Transform(trainLsaView);
var predTestLsa  = lsaModel.Transform(testLsaView);

var metricsTrainLsa = mlContext.BinaryClassification.Evaluate(predTrainLsa, labelColumnName: nameof(ModelInput.Label));
var metricsTestLsa  = mlContext.BinaryClassification.Evaluate(predTestLsa, labelColumnName: nameof(ModelInput.Label));

Console.WriteLine("LSA — train metrics:");
Console.WriteLine($"  Accuracy: {metricsTrainLsa.Accuracy:F4}, F1: {metricsTrainLsa.F1Score:F4}, AUC: {metricsTrainLsa.AreaUnderRocCurve:F4}");
Console.WriteLine("LSA — test metrics:");
Console.WriteLine($"  Accuracy: {metricsTestLsa.Accuracy:F4}, F1: {metricsTestLsa.F1Score:F4}, AUC: {metricsTestLsa.AreaUnderRocCurve:F4}");

public class FeatureRow
{
    public bool Label { get; set; }
    public VBuffer<float> Features { get; set; }
}

public class FeatureWithPca : FeatureRow
{
    public VBuffer<float> FeaturesPca { get; set; }
}

var lsaTrainVectors = mlContext.Data.CreateEnumerable<FeatureRow>(trainLsaView, reuseRowObject: false)
    .Select(r => ToDense(r.Features))
    .ToArray();
var lsaTestVectors = mlContext.Data.CreateEnumerable<FeatureRow>(testLsaView, reuseRowObject: false)
    .Select(r => ToDense(r.Features))
    .ToArray();

var lsaTrainPca = mlContext.Data.CreateEnumerable<FeatureWithPca>(trainLsaView, reuseRowObject: false)
    .ToArray();


LSA — train metrics:
  Accuracy: 0.8468, F1: 0.8560, AUC: 0.9224
LSA — test metrics:
  Accuracy: 0.8275, F1: 0.8354, AUC: 0.9103


### 4. Word2Vec: Skip-gram + Negative Sampling

Реализуем обучение skip-gram на C#: формируем словарь, рассчитываем распределение для негативного сэмплирования и итеративно обновляем веса.
Документ представляем усреднением словарных векторов, после чего снова обучаем логистическую регрессию и сравниваем качество с LSA.


In [18]:
Dictionary<string, int> wordToId = default!;
double[] negativeSampler = Array.Empty<double>();
int[][] docTokenIds = Array.Empty<int[]>();
int[][] docTokenIdsSampled = Array.Empty<int[]>();

try
{
    var tokenCounts = new Dictionary<string, int>(StringComparer.Ordinal);
    foreach (var doc in processedRecords)
    {
        foreach (var token in doc.Tokens)
        {
            if (!tokenCounts.TryGetValue(token, out var cnt)) cnt = 0;
            tokenCounts[token] = cnt + 1;
        }
    }

    int minCount = 5;
    var vocab = tokenCounts
        .Where(kv => kv.Value >= minCount)
        .OrderByDescending(kv => kv.Value)
        .Select((kv, idx) => new { Token = kv.Key, Count = kv.Value, Index = idx })
        .ToArray();

    wordToId = vocab.ToDictionary(v => v.Token, v => v.Index, StringComparer.Ordinal);

    Console.WriteLine($"Word2Vec vocab size: {wordToId.Count}");

    var freqPow = vocab.Select(v => Math.Pow(v.Count, 0.75)).ToArray();
    double totalPow = freqPow.Sum();
    negativeSampler = new double[freqPow.Length];
    double accum = 0;
    for (int i = 0; i < negativeSampler.Length; i++)
    {
        accum += freqPow[i] / totalPow;
        negativeSampler[i] = accum;
    }
    if (negativeSampler.Length > 0)
        negativeSampler[^1] = 1.0;

    docTokenIds = processedRecords
        .Select(doc => doc.Tokens
            .Select(tok => wordToId.TryGetValue(tok, out var id) ? id : -1)
            .Where(id => id >= 0)
            .ToArray())
        .ToArray();

    int maxTokensPerDoc = 80;
    var rngSubsample = new Random(2024);
    docTokenIdsSampled = docTokenIds
        .Select(tokens =>
        {
            var indices = SampleIndices(tokens.Length, maxTokensPerDoc, rngSubsample);
            return indices.Select(idx => tokens[idx]).ToArray();
        })
        .ToArray();

    int embeddingSize = 128;
    word2VecDimension = embeddingSize;
    int window = 4;
    int negativeSamples = 4;
    int word2VecEpochs = 2;
    double initialLr = 0.025;

    var rngWord2Vec = new Random(42);

    var inputEmbeddings = new float[wordToId.Count][];
    var outputEmbeddings = new float[wordToId.Count][];
    for (int i = 0; i < wordToId.Count; i++)
    {
        inputEmbeddings[i] = new float[embeddingSize];
        outputEmbeddings[i] = new float[embeddingSize];
        for (int d = 0; d < embeddingSize; d++)
        {
            inputEmbeddings[i][d] = (float)((rngWord2Vec.NextDouble() - 0.5) / embeddingSize);
            outputEmbeddings[i][d] = 0f;
        }
    }

    var trainingDocIndices = docTokenIdsSampled
        .Select((tokens, idx) => (tokens, idx))
        .Where(item => item.tokens.Length >= 2)
        .Select(item => item.idx)
        .ToList();

    word2VecInputEmbeddings = inputEmbeddings;
    word2VecOutputEmbeddings = outputEmbeddings;
    if (trainingDocIndices.Count > 8000)
    {
        Shuffle(trainingDocIndices, rngWord2Vec);
        trainingDocIndices = trainingDocIndices.Take(8000).ToList();
    }

    for (int epoch = 0; epoch < word2VecEpochs; epoch++)
    {
        var order = trainingDocIndices.ToList();
        Shuffle(order, rngWord2Vec);

        double loss = 0;
        double lr = initialLr * (1.0 - (double)epoch / Math.Max(1, word2VecEpochs));
        lr = Math.Max(lr, initialLr * 0.25);

        foreach (var docIdx in order)
        {
            var tokens = docTokenIdsSampled[docIdx];
            if (tokens.Length < 2)
                continue;

            for (int pos = 0; pos < tokens.Length; pos++)
            {
                int centerId = tokens[pos];
                if (centerId < 0) continue;

                int currentWindow = rngWord2Vec.Next(1, window + 1);
                int start = Math.Max(0, pos - currentWindow);
                int end = Math.Min(tokens.Length - 1, pos + currentWindow);

                for (int ctx = start; ctx <= end; ctx++)
                {
                    if (ctx == pos) continue;
                    int contextId = tokens[ctx];
                    if (contextId < 0) continue;

                    var centerVec = inputEmbeddings[centerId];
                    var contextVec = outputEmbeddings[contextId];

                    double score = 0;
                    for (int d = 0; d < embeddingSize; d++)
                        score += centerVec[d] * contextVec[d];

                    double sig = Sigmoid(score);
                    double grad = (1.0 - sig) * lr;

                    for (int d = 0; d < embeddingSize; d++)
                    {
                        float cVal = centerVec[d];
                        float oVal = contextVec[d];
                        float deltaCenter = (float)(grad * oVal);
                        float deltaContext = (float)(grad * cVal);
                        centerVec[d] += deltaCenter;
                        contextVec[d] += deltaContext;
                    }
                    loss -= Math.Log(sig + 1e-8);

                    for (int n = 0; n < negativeSamples; n++)
                    {
                        int negativeId = SampleNegative(rngWord2Vec, negativeSampler);
                        if (negativeId == centerId || negativeId == contextId)
                            continue;

                        var negativeVec = outputEmbeddings[negativeId];
                        double negScore = 0;
                        for (int d = 0; d < embeddingSize; d++)
                            negScore += centerVec[d] * negativeVec[d];

                        double negSig = Sigmoid(negScore);
                        double negGrad = (-negSig) * lr;

                        for (int d = 0; d < embeddingSize; d++)
                        {
                            float cVal = centerVec[d];
                            float nVal = negativeVec[d];
                            float deltaCenter = (float)(negGrad * nVal);
                            float deltaNeg = (float)(negGrad * cVal);
                            centerVec[d] += deltaCenter;
                            negativeVec[d] += deltaNeg;
                        }

                        loss -= Math.Log(1.0 - negSig + 1e-8);
                    }
                }
            }
        }

        Console.WriteLine($"Word2Vec epoch {epoch + 1}/{word2VecEpochs} — loss ≈ {loss:F2}, lr={lr:F4}");
    }

    float[] BuildWord2VecDoc(int[] tokenIds)
    {
        var vector = new float[embeddingSize];
        int count = 0;
        foreach (var id in tokenIds)
        {
            if (id < 0 || id >= inputEmbeddings.Length) continue;
            var embed = inputEmbeddings[id];
            for (int d = 0; d < embeddingSize; d++)
                vector[d] += embed[d];
            count++;
        }
        if (count == 0)
            return vector;
        float inv = 1f / count;
        for (int d = 0; d < embeddingSize; d++)
            vector[d] *= inv;
        NormalizeInPlace(vector);
        return vector;
    }

    var word2VecDocVectors = docTokenIds.Select(BuildWord2VecDoc).ToArray();
    word2VecTrainMatrix = trainRecords.Select(r => word2VecDocVectors[r.GlobalId]).ToArray();
    word2VecTestMatrix  = testRecords.Select(r => word2VecDocVectors[r.GlobalId]).ToArray();

    var word2VecTrainExamples = trainRecords
        .Select(r => new VectorExample { Label = r.Label, Features = Clone(word2VecDocVectors[r.GlobalId]) })
        .ToList();
    var word2VecTestExamples = testRecords
        .Select(r => new VectorExample { Label = r.Label, Features = Clone(word2VecDocVectors[r.GlobalId]) })
        .ToList();

    var word2VecTrainView = mlContext.Data.LoadFromEnumerable(word2VecTrainExamples);
    var word2VecTestView  = mlContext.Data.LoadFromEnumerable(word2VecTestExamples);

    var word2VecModelLocal = embeddingTrainer.Fit(word2VecTrainView);
    var predTrainWord2Vec = word2VecModelLocal.Transform(word2VecTrainView);
    var predTestWord2Vec  = word2VecModelLocal.Transform(word2VecTestView);
    word2VecModel = word2VecModelLocal;
    predTrainWord2VecView = predTrainWord2Vec;
    predTestWord2VecView  = predTestWord2Vec;

    metricsTrainWord2Vec = mlContext.BinaryClassification.Evaluate(predTrainWord2Vec, labelColumnName: nameof(VectorExample.Label));
    metricsTestWord2Vec  = mlContext.BinaryClassification.Evaluate(predTestWord2Vec, labelColumnName: nameof(VectorExample.Label));

    Console.WriteLine("Word2Vec — train metrics:");
    Console.WriteLine($"  Accuracy: {metricsTrainWord2Vec.Accuracy:F4}, F1: {metricsTrainWord2Vec.F1Score:F4}, AUC: {metricsTrainWord2Vec.AreaUnderRocCurve:F4}");
    Console.WriteLine("Word2Vec — test metrics:");
    Console.WriteLine($"  Accuracy: {metricsTestWord2Vec.Accuracy:F4}, F1: {metricsTestWord2Vec.F1Score:F4}, AUC: {metricsTestWord2Vec.AreaUnderRocCurve:F4}");
}
catch (Exception ex)
{
    Console.WriteLine($"Word2Vec error: {ex}");
    throw;
}


Word2Vec vocab size: 1346
Word2Vec epoch 1/2 — loss ≈ 345883.63, lr=0.0250
Word2Vec epoch 2/2 — loss ≈ 281677.11, lr=0.0125
Word2Vec — train metrics:
  Accuracy: 0.6727, F1: 0.6624, AUC: 0.7513
Word2Vec — test metrics:
  Accuracy: 0.6870, F1: 0.6735, AUC: 0.7521


<null>

### 5. Doc2Vec (Distributed Bag of Words)

Документные вектора обучаются совместно с матрицей выходных весов слов: контекст задаётся как «мешок» токенов, а обновления выполняются через отрицательные примеры.
Нормализованные Doc2Vec-вектора подаются в ту же логистическую регрессию, что и Word2Vec, чтобы оценка шла в идентичной постановке.


In [19]:
try
{
    int doc2VecDim = 128;
    doc2VecDimension = doc2VecDim;
    int doc2VecEpochs = 3;
    int doc2VecNegative = 4;
    doc2VecNegativeSamples = doc2VecNegative;
    double doc2VecLr = 0.05;
    doc2VecInitialLearningRate = doc2VecLr;

    var docEmbeddings = new float[processedRecords.Count][];
    var docWordOutput = new float[wordToId.Count][];
    var rngDoc2Vec = new Random(99);

    for (int i = 0; i < docEmbeddings.Length; i++)
    {
        docEmbeddings[i] = new float[doc2VecDim];
        for (int d = 0; d < doc2VecDim; d++)
            docEmbeddings[i][d] = (float)((rngDoc2Vec.NextDouble() - 0.5) / doc2VecDim);
    }
    for (int i = 0; i < docWordOutput.Length; i++)
    {
        docWordOutput[i] = new float[doc2VecDim];
        for (int d = 0; d < doc2VecDim; d++)
            docWordOutput[i][d] = (float)((rngDoc2Vec.NextDouble() - 0.5) / doc2VecDim);
    }

    doc2VecTrainedEmbeddings = docEmbeddings;
    doc2VecWordOutput = docWordOutput;
    for (int epoch = 0; epoch < doc2VecEpochs; epoch++)
    {
        var order = Enumerable.Range(0, processedRecords.Count).ToList();
        Shuffle(order, rngDoc2Vec);

        double loss = 0;
        double lr = doc2VecLr * (1.0 - (double)epoch / Math.Max(1, doc2VecEpochs));
        lr = Math.Max(lr, doc2VecLr * 0.3);

        foreach (var docIdx in order)
        {
            var tokens = docTokenIdsSampled[docIdx];
            if (tokens.Length == 0)
                continue;

            var docVec = docEmbeddings[docIdx];

            foreach (var wordId in tokens)
            {
                var wordVec = docWordOutput[wordId];
                double score = 0;
                for (int d = 0; d < doc2VecDim; d++)
                    score += docVec[d] * wordVec[d];

                double sig = Sigmoid(score);
                double grad = (1.0 - sig) * lr;

                for (int d = 0; d < doc2VecDim; d++)
                {
                    float dv = docVec[d];
                    float wv = wordVec[d];
                    float deltaDoc = (float)(grad * wv);
                    float deltaWord = (float)(grad * dv);
                    docVec[d] += deltaDoc;
                    wordVec[d] += deltaWord;
                }
                loss -= Math.Log(sig + 1e-8);

                for (int n = 0; n < doc2VecNegative; n++)
                {
                    int negativeId = SampleNegative(rngDoc2Vec, negativeSampler);
                    if (negativeId == wordId)
                        continue;

                    var negativeVec = docWordOutput[negativeId];
                    double negScore = 0;
                    for (int d = 0; d < doc2VecDim; d++)
                        negScore += docVec[d] * negativeVec[d];

                    double negSig = Sigmoid(negScore);
                    double negGrad = (-negSig) * lr;

                    for (int d = 0; d < doc2VecDim; d++)
                    {
                        float dv = docVec[d];
                        float nv = negativeVec[d];
                        float deltaDoc = (float)(negGrad * nv);
                        float deltaNeg = (float)(negGrad * dv);
                        docVec[d] += deltaDoc;
                        negativeVec[d] += deltaNeg;
                    }

                    loss -= Math.Log(1.0 - negSig + 1e-8);
                }
            }
        }

        Console.WriteLine($"Doc2Vec epoch {epoch + 1}/{doc2VecEpochs} — loss ≈ {loss:F2}, lr={lr:F4}");
    }

    foreach (var docVec in docEmbeddings)
        NormalizeInPlace(docVec);

    doc2VecTrainMatrix = trainRecords.Select(r => docEmbeddings[r.GlobalId]).ToArray();
    doc2VecTestMatrix  = testRecords.Select(r => docEmbeddings[r.GlobalId]).ToArray();
var doc2VecTrainExamples = trainRecords
        .Select(r => new VectorExample { Label = r.Label, Features = Clone(docEmbeddings[r.GlobalId]) })
        .ToList();
    var doc2VecTestExamples = testRecords
        .Select(r => new VectorExample { Label = r.Label, Features = Clone(docEmbeddings[r.GlobalId]) })
        .ToList();

    var doc2VecTrainView = mlContext.Data.LoadFromEnumerable(doc2VecTrainExamples);
    var doc2VecTestView  = mlContext.Data.LoadFromEnumerable(doc2VecTestExamples);

    var doc2VecModelLocal = embeddingTrainer.Fit(doc2VecTrainView);
    var predTrainDoc2Vec = doc2VecModelLocal.Transform(doc2VecTrainView);
    var predTestDoc2Vec  = doc2VecModelLocal.Transform(doc2VecTestView);
    doc2VecModel = doc2VecModelLocal;
    predTrainDoc2VecView = predTrainDoc2Vec;
    predTestDoc2VecView  = predTestDoc2Vec;

    metricsTrainDoc2Vec = mlContext.BinaryClassification.Evaluate(predTrainDoc2Vec, labelColumnName: nameof(VectorExample.Label));
    metricsTestDoc2Vec  = mlContext.BinaryClassification.Evaluate(predTestDoc2Vec, labelColumnName: nameof(VectorExample.Label));

    Console.WriteLine("Doc2Vec — train metrics:");
    Console.WriteLine($"  Accuracy: {metricsTrainDoc2Vec.Accuracy:F4}, F1: {metricsTrainDoc2Vec.F1Score:F4}, AUC: {metricsTrainDoc2Vec.AreaUnderRocCurve:F4}");
    Console.WriteLine("Doc2Vec — test metrics:");
    Console.WriteLine($"  Accuracy: {metricsTestDoc2Vec.Accuracy:F4}, F1: {metricsTestDoc2Vec.F1Score:F4}, AUC: {metricsTestDoc2Vec.AreaUnderRocCurve:F4}");
}
catch (Exception ex)
{
    Console.WriteLine($"Doc2Vec error: {ex}");
    throw;
}


Doc2Vec epoch 1/3 — loss ≈ 110978.34, lr=0.0500
Doc2Vec epoch 2/3 — loss ≈ 110954.36, lr=0.0333
Doc2Vec epoch 3/3 — loss ≈ 110957.26, lr=0.0167
Doc2Vec — train metrics:
  Accuracy: 0.5940, F1: 0.6617, AUC: 0.6226
Doc2Vec — test metrics:
  Accuracy: 0.5487, F1: 0.6211, AUC: 0.5551


### 6. Сравнение метрик

Сводим Accuracy, F1 и AUC на одном тестовом множестве. Это первый уровень сравнения представлений, позволяющий быстро выявить переобучение и разницу в качестве.


In [20]:
Console.WriteLine("\nМетрики (test):");
Console.WriteLine("Method      Acc     F1     AUC");
Console.WriteLine($"LSA        {metricsTestLsa.Accuracy,6:F3}  {metricsTestLsa.F1Score,6:F3}  {metricsTestLsa.AreaUnderRocCurve,6:F3}");
if (metricsTestWord2Vec is not null)
    Console.WriteLine($"Word2Vec    {metricsTestWord2Vec.Accuracy,6:F3}  {metricsTestWord2Vec.F1Score,6:F3}  {metricsTestWord2Vec.AreaUnderRocCurve,6:F3}");
else
    Console.WriteLine("Word2Vec    n/a");
if (metricsTestDoc2Vec is not null)
    Console.WriteLine($"Doc2Vec     {metricsTestDoc2Vec.Accuracy,6:F3}  {metricsTestDoc2Vec.F1Score,6:F3}  {metricsTestDoc2Vec.AreaUnderRocCurve,6:F3}");
else
    Console.WriteLine("Doc2Vec     n/a");



Метрики (test):
Method      Acc     F1     AUC
LSA         0.827   0.835   0.910
Word2Vec     0.687   0.674   0.752
Doc2Vec      0.549   0.621   0.555


### 6.1. Примеры предсказаний на одной выборке

Смотрим, как различные векторные представления классифицируют одни и те же отзывы.
Так можно дополнить числовые метрики конкретными наблюдениями по ошибкам и уверенностям.


In [21]:
public class PredictionRow
{
    public bool PredictedLabel { get; set; }
    public float Probability { get; set; }
    public float Score { get; set; }
}

PredictionRow[] ExtractPredictions(IDataView? view)
{
    return view is null
        ? Array.Empty<PredictionRow>()
        : mlContext.Data.CreateEnumerable<PredictionRow>(view, reuseRowObject: false).ToArray();
}

var lsaPredRows = ExtractPredictions(predTestLsa);
var word2VecPredRows = ExtractPredictions(predTestWord2VecView);
var doc2VecPredRows = ExtractPredictions(predTestDoc2VecView);

var sampleIndices = new[] { 0, Math.Min(5, testRecords.Count - 1), Math.Max(0, testRecords.Count - 1) }
    .Distinct()
    .Where(idx => idx >= 0 && idx < testRecords.Count)
    .ToArray();

void PrintPrediction(string method, PredictionRow[] rows, int idx)
{
    if (rows.Length <= idx)
    {
        Console.WriteLine($"  {method}: нет предсказаний");
        return;
    }
    var pred = rows[idx];
    var label = pred.PredictedLabel ? "positive" : "negative";
    Console.WriteLine($"  {method}: {label} (p={pred.Probability:F3}, score={pred.Score:F3})");
}

foreach (var idx in sampleIndices)
{
    var rec = testRecords[idx];
    Console.WriteLine($"\nДокумент #{idx} — истинная метка: {rec.Sentiment}");
    Console.WriteLine(Truncate(rec.Text));
    PrintPrediction("LSA", lsaPredRows, idx);
    PrintPrediction("Word2Vec", word2VecPredRows, idx);
    PrintPrediction("Doc2Vec", doc2VecPredRows, idx);
}



Документ #0 — истинная метка: positive
Shanghai is also really exciting (precisely -- skyscrapers galore). Good tweeps in China:  (SH)  (BJ).
  LSA: positive (p=0.829, score=1.582)
  Word2Vec: positive (p=0.527, score=0.107)
  Doc2Vec: positive (p=0.558, score=0.234)

Документ #5 — истинная метка: negative
My bike was put on hold...should have known that.... argh total bummer
  LSA: negative (p=0.416, score=-0.338)
  Word2Vec: negative (p=0.409, score=-0.368)
  Doc2Vec: negative (p=0.437, score=-0.255)

Документ #1929 — истинная метка: positive
http://twitpic.com/4woj2 - omgssh  ang cute ng bby.!
  LSA: positive (p=0.663, score=0.676)
  Word2Vec: negative (p=0.478, score=-0.089)
  Doc2Vec: positive (p=0.556, score=0.227)


### 7. Поиск похожих документов

Для выбранного тестового отзыва находим несколько ближайших обучающих документов по косинусному сходству в пространствах LSA, Word2Vec и Doc2Vec.
Так можно наглядно увидеть, какие темы и формулировки лучше ловит каждое представление.


In [22]:
int exampleIndex = Math.Min(2, testRecords.Count - 1);
var queryDoc = testRecords[exampleIndex];

Console.WriteLine("Запрос (test):");
Console.WriteLine(queryDoc.Text);
Console.WriteLine($"Clean → {queryDoc.CleanText}");
Console.WriteLine($"Sentiment: {queryDoc.Sentiment}");

var lsaQuery = lsaTestVectors[exampleIndex];
var word2VecQuery = word2VecTestMatrix[exampleIndex];
var doc2VecQuery = doc2VecTestMatrix[exampleIndex];

void PrintNeighbours(string title, float[] query, IReadOnlyList<float[]> matrix)
{
    Console.WriteLine($"\n{title}");
    var neighbours = GetTopSimilar(query, matrix, 5);
    foreach (var (idx, score) in neighbours)
    {
        var doc = trainRecords[idx];
        Console.WriteLine($"Score {score:F3} | {doc.Sentiment}");
        Console.WriteLine(Truncate(doc.Text));
        Console.WriteLine("---");
    }
}

PrintNeighbours("LSA", lsaQuery, lsaTrainVectors);
PrintNeighbours("Word2Vec", word2VecQuery, word2VecTrainMatrix);
PrintNeighbours("Doc2Vec", doc2VecQuery, doc2VecTrainMatrix);


Запрос (test):
that`s great!! weee!! visitors!
Clean → that`s great weee visitors
Sentiment: positive

LSA
Score 0.862 | positive
That`s a great idea
---
Score 0.605 | positive
That`s what`s up hah. I`m great, thanks  Just waiting for all the craziness to take off.
---
Score 0.533 | positive
no school today! that`s greeeeeat!
---
Score 0.520 | positive
Reds win! Great end to a great day
---
Score 0.509 | positive
that`s a good attitude
---

Word2Vec
Score 1.000 | positive
heyy i finally got one too  oh and good luck on your finals today
---
Score 1.000 | positive
the commies at their finest: youtube and blogger are blocked in china. no updates from us while we are in china.  but GREAT WALL CONQUERED
---
Score 1.000 | positive
def cheese and onion  however after being back in the states for 4 months, finding a bag of salt&vinegar...they tasted GREAT!
---
Score 1.000 | positive
it was a great wedding! the band was awesome (they played a ton of great 80`s songs) as was the food!
---
Score

### 8. Визуализация

Строим кривую накопленного вклада PCA для LSA и проекции в двумерное пространство для всех трёх подходов.
Сравниваем компактность и разделимость классов, чтобы дополнить числовые метрики наглядной картиной.


In [23]:
// LSA: cumulative explained variance
int components = Math.Min(lsaRank, lsaTrainPca.Length > 0 ? lsaTrainPca[0].FeaturesPca.Length : 0);
var variance = new double[components];
var mean = new double[components];
long count = 0;
foreach (var row in lsaTrainPca.Take(5000))
{
    var dense = ToDense(row.FeaturesPca);
    count++;
    for (int i = 0; i < components; i++)
    {
        double val = dense[i];
        mean[i] += val;
        variance[i] += val * val;
    }
}
if (count > 0)
{
    for (int i = 0; i < components; i++)
    {
        mean[i] /= count;
        variance[i] = variance[i] / count - mean[i] * mean[i];
        if (variance[i] < 0) variance[i] = 0;
    }
}
var totalVar = variance.Sum();
var cumulative = new double[components];
double accVar = 0;
for (int i = 0; i < components; i++)
{
    double share = totalVar > 0 ? variance[i] / totalVar : 0;
    accVar += share;
    cumulative[i] = accVar * 100.0;
}

var pltVariance = new ScottPlot.Plot();
var xsVar = Enumerable.Range(1, components).Select(i => (double)i).ToArray();
pltVariance.Add.Scatter(xsVar, cumulative);
Console.WriteLine($"Variance curve prepared (components={components})");

public class TwoDimRow
{
    public bool Label { get; set; }
    public VBuffer<float> PC { get; set; }
}

TwoDimRow[] ProjectTo2D(float[][] matrix, IList<ProcessedRecord> records)
{
    try
    {
        if (matrix.Length == 0 || records.Count == 0)
            return Array.Empty<TwoDimRow>();

        int vectorSize = matrix[0].Length;
        if (vectorSize < 2)
            return Array.Empty<TwoDimRow>();

        int count = Math.Min(matrix.Length, records.Count);
        if (vectorSize == 128)
        {
            var data128 = records.Take(count).Select((rec, idx) => new VectorExample
            {
                Label = rec.Label,
                Features = Clone(matrix[idx])
            });
            var view128 = mlContext.Data.LoadFromEnumerable(data128);
            var pca128 = mlContext.Transforms.ProjectToPrincipalComponents("PC", nameof(VectorExample.Features), rank: 2).Fit(view128);
            return mlContext.Data.CreateEnumerable<TwoDimRow>(pca128.Transform(view128), reuseRowObject: false).ToArray();
        }

        var dataLsa = records.Take(count).Select((rec, idx) => new LsaVectorExample
        {
            Label = rec.Label,
            Features = Clone(matrix[idx])
        });
        var viewLsa = mlContext.Data.LoadFromEnumerable(dataLsa);
        var pcaLsa = mlContext.Transforms.ProjectToPrincipalComponents("PC", nameof(LsaVectorExample.Features), rank: 2).Fit(viewLsa);
        return mlContext.Data.CreateEnumerable<TwoDimRow>(pcaLsa.Transform(viewLsa), reuseRowObject: false).ToArray();
    }
    catch (Exception ex)
    {
        Console.WriteLine($"ProjectTo2D error (records={records.Count}, matrix={matrix.Length}): {ex.Message}");
        return Array.Empty<TwoDimRow>();
    }
}

var lsa2D = ProjectTo2D(lsaTrainVectors, trainRecords);
var word2Vec2D = ProjectTo2D(word2VecTrainMatrix, trainRecords);
var doc2Vec2D = ProjectTo2D(doc2VecTrainMatrix, trainRecords);

void DescribeScatter(TwoDimRow[] rows, string title)
{
    var samplePos = rows.Where(r => r.Label).Take(5).Select(r => ToDense(r.PC)).ToArray();
    var sampleNeg = rows.Where(r => !r.Label).Take(5).Select(r => ToDense(r.PC)).ToArray();
    double avgPosX = samplePos.Length > 0 ? samplePos.Average(v => v[0]) : 0;
    double avgNegX = sampleNeg.Length > 0 ? sampleNeg.Average(v => v[0]) : 0;
    Console.WriteLine($"{title}: points={rows.Length}, avg PC1 pos={avgPosX:F3}, avg PC1 neg={avgNegX:F3}");
}

DescribeScatter(lsa2D, "LSA 2D");
DescribeScatter(word2Vec2D, "Word2Vec 2D");
DescribeScatter(doc2Vec2D, "Doc2Vec 2D");


Variance curve prepared (components=300)
LSA 2D: points=4000, avg PC1 pos=-0.077, avg PC1 neg=0.081
Word2Vec 2D: points=4000, avg PC1 pos=-0.007, avg PC1 neg=0.071
Doc2Vec 2D: points=4000, avg PC1 pos=0.024, avg PC1 neg=0.029


### 9. Пользовательские проверки (ручной ввод)

В конце можно указать любые строки в массиве `customTexts`, чтобы прогнать их через предобработку
и три обученных анализатора. Так проверяется, как модели реагируют на новые пользовательские данные.


In [24]:
string[] customTexts =
{
    "Очень понравился новый интерфейс приложения, все работает быстро и без сбоев.",
    "Качество обслуживания ужасное: половину заказа забыли и даже не извинились."
};

Console.WriteLine("=== Пользовательские тексты ===");
if (customTexts.Length == 0)
{
    Console.WriteLine("Добавьте примеры в массив customTexts и перезапустите ячейку.");
}
else
{
    var raw = customTexts.Select((text, idx) => new SentimentRecord
    {
        DocId = idx,
        Text = text,
        Sentiment = "custom",
        Label = false,
        Split = "custom"
    }).ToList();

    var customView = mlContext.Data.LoadFromEnumerable(raw);
    var tokenView = textTransformer.Transform(customView);
    var tokenized = mlContext.Data.CreateEnumerable<TokenizedRecord>(tokenView, reuseRowObject: false).ToList();

    var skipped = new List<int>();
    var prepared = new List<(ProcessedRecord Record, int InputIndex)>();

    foreach (var item in tokenized.Select((rec, idx) => (rec, idx)))
    {
        var tokens = (item.rec.Tokens ?? Array.Empty<ReadOnlyMemory<char>>())
            .Select(t => t.ToString())
            .Where(t => !string.IsNullOrWhiteSpace(t) && t.Length > 1)
            .ToArray();

        if (tokens.Length < 3)
        {
            skipped.Add(item.idx);
            continue;
        }

        prepared.Add((new ProcessedRecord
        {
            Text = item.rec.Text ?? string.Empty,
            CleanText = string.Join(" ", tokens),
            Sentiment = "custom",
            Tokens = tokens,
            Label = false,
            SplitIndex = item.idx,
            GlobalId = item.idx
        }, item.idx));
    }

    if (skipped.Count > 0)
        Console.WriteLine($"Предупреждение: пропущено {skipped.Count} строк(и) после очистки из-за малого числа токенов.");

    if (prepared.Count == 0)
    {
        Console.WriteLine("После предобработки не осталось текстов. Измените строки в customTexts.");
    }
    else
    {
        int count = prepared.Count;
        var ordered = prepared.Select(p => p.Record).ToList();
        var tokenIds = ordered
            .Select(r => r.Tokens
                .Select(tok => wordToId.TryGetValue(tok, out var id) ? id : -1)
                .Where(id => id >= 0)
                .ToArray())
            .ToList();

        float[] BuildWord2VecVector(int[] ids)
        {
            int dim = word2VecDimension > 0 ? word2VecDimension : 0;
            var vector = dim > 0 ? new float[dim] : Array.Empty<float>();
            if (ids.Length == 0 || word2VecInputEmbeddings.Length == 0 || vector.Length == 0)
                return vector;
            int used = 0;
            foreach (var id in ids)
            {
                if (id < 0 || id >= word2VecInputEmbeddings.Length)
                    continue;
                var embed = word2VecInputEmbeddings[id];
                for (int d = 0; d < vector.Length; d++)
                    vector[d] += embed[d];
                used++;
            }
            if (used == 0)
                return vector;
            float inv = 1f / used;
            for (int d = 0; d < vector.Length; d++)
                vector[d] *= inv;
            NormalizeInPlace(vector);
            return vector;
        }

        float[] InferDoc2VecVector(int[] ids, int epochs = 30)
        {
            int dim = doc2VecDimension > 0 ? doc2VecDimension : 0;
            var vector = dim > 0 ? new float[dim] : Array.Empty<float>();
            if (ids.Length == 0 || doc2VecWordOutput.Length == 0 || vector.Length == 0)
                return vector;

            var rng = new Random(1000 + ids.Length);
            for (int d = 0; d < vector.Length; d++)
                vector[d] = (float)((rng.NextDouble() - 0.5) / Math.Max(1, vector.Length));

            double baseLr = doc2VecInitialLearningRate > 0 ? doc2VecInitialLearningRate : 0.05;
            int negatives = doc2VecNegativeSamples > 0 ? doc2VecNegativeSamples : 4;

            for (int epoch = 0; epoch < epochs; epoch++)
            {
                double lr = baseLr * (1.0 - (double)epoch / Math.Max(1, epochs));
                lr = Math.Max(lr, baseLr * 0.3);

                foreach (var wordId in ids)
                {
                    if (wordId < 0 || wordId >= doc2VecWordOutput.Length)
                        continue;
                    var wordVec = doc2VecWordOutput[wordId];
                    double score = 0;
                    for (int d = 0; d < vector.Length; d++)
                        score += vector[d] * wordVec[d];
                    double sig = Sigmoid(score);
                    double grad = (1.0 - sig) * lr;
                    for (int d = 0; d < vector.Length; d++)
                        vector[d] += (float)(grad * wordVec[d]);

                    if (negativeSampler.Length == 0)
                        continue;
                    for (int n = 0; n < negatives; n++)
                    {
                        int negativeId = SampleNegative(rng, negativeSampler);
                        if (negativeId < 0 || negativeId >= doc2VecWordOutput.Length || negativeId == wordId)
                            continue;
                        var negVec = doc2VecWordOutput[negativeId];
                        double negScore = 0;
                        for (int d = 0; d < vector.Length; d++)
                            negScore += vector[d] * negVec[d];
                        double negSig = Sigmoid(negScore);
                        double negGrad = (-negSig) * lr;
                        for (int d = 0; d < vector.Length; d++)
                            vector[d] += (float)(negGrad * negVec[d]);
                    }
                }
            }

            NormalizeInPlace(vector);
            return vector;
        }

        PredictionRow[] lsaRows = Array.Empty<PredictionRow>();
        if (lsaTransformer is not null)
        {
            var inputs = ordered.Select(r => new ModelInput { Text = r.CleanText, Label = false }).ToList();
            var view = mlContext.Data.LoadFromEnumerable(inputs);
            var features = lsaTransformer.Transform(view);
            var scored = lsaModel.Transform(features);
            lsaRows = ExtractPredictions(scored);
        }

        PredictionRow[] word2VecRows = Array.Empty<PredictionRow>();
        if (word2VecModel is not null && word2VecDimension > 0)
        {
            var examples = tokenIds.Select(ids => new VectorExample
            {
                Label = false,
                Features = BuildWord2VecVector(ids)
            }).ToList();
            var view = mlContext.Data.LoadFromEnumerable(examples);
            word2VecRows = ExtractPredictions(word2VecModel.Transform(view));
        }

        PredictionRow[] doc2VecRows = Array.Empty<PredictionRow>();
        if (doc2VecModel is not null && doc2VecDimension > 0)
        {
            var examples = tokenIds.Select(ids => new VectorExample
            {
                Label = false,
                Features = InferDoc2VecVector(ids)
            }).ToList();
            var view = mlContext.Data.LoadFromEnumerable(examples);
            doc2VecRows = ExtractPredictions(doc2VecModel.Transform(view));
        }

        for (int i = 0; i < prepared.Count; i++)
        {
            var (record, idx) = prepared[i];
            Console.WriteLine();
            Console.WriteLine($"Пример {i + 1} (строка {idx + 1}):");
            Console.WriteLine(Truncate(customTexts[idx], 220));
            Console.WriteLine($"Clean → {record.CleanText}");
            PrintPrediction("LSA", lsaRows, i);
            PrintPrediction("Word2Vec", word2VecRows, i);
            PrintPrediction("Doc2Vec", doc2VecRows, i);
        }
    }
}


=== Пользовательские тексты ===

Пример 1 (строка 1):
Очень понравился новый интерфейс приложения, все работает быстро и без сбоев.
Clean → очень понравился новыи интерфеис приложения все работает быстро без сбоев
  LSA: negative (p=0.453, score=-0.188)
  Word2Vec: negative (p=0.305, score=-0.822)
  Doc2Vec: positive (p=0.548, score=0.194)

Пример 2 (строка 2):
Качество обслуживания ужасное: половину заказа забыли и даже не извинились.
Clean → качество обслуживания ужасное половину заказа забыли даже не извинились
  LSA: negative (p=0.453, score=-0.188)
  Word2Vec: negative (p=0.305, score=-0.822)
  Doc2Vec: positive (p=0.548, score=0.194)
